In [11]:
import warnings
warnings.filterwarnings("ignore")

import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
import matplotlib.colors as mcolors

### Data Loading

In [118]:
df= pd.read_csv("STOFSatl_hydro.csv")

df.head()

,time_UTC,observed_data,forecast_data,offset,x,y,station_id,agency,storm,year,time_UTC_dt
0,2004-08-09 00:00:00,NaN,NaN,NaN,-79.962864,32.890452,21720677,USGS,CHARLEY,2004,0.0
1,2004-08-09 01:00:00,-1.874,NaN,NaN,-79.962864,32.890452,21720677,USGS,CHARLEY,2004,60.0
2,2004-08-09 02:00:00,-1.964,NaN,NaN,-79.962864,32.890452,21720677,USGS,CHARLEY,2004,120.0
3,2004-08-09 03:00:00,-1.554,NaN,NaN,-79.962864,32.890452,21720677,USGS,CHARLEY,2004,180.0
4,2004-08-09 04:00:00,-0.674,NaN,NaN,-79.962864,32.890452,21720677,USGS,CHARLEY,2004,240.0


In [119]:
df.dropna(inplace= True)

In [120]:
df['station_id']= df['station_id'].astype(str)

In [121]:
df['storm'].value_counts()

storm
HARVEY     82834
IDALIA     61251
HERMINE    51887
IAN        41491
MATTHEW    37523
IDA        34993
CHARLEY    17935
Name: count, dtype: int64

In [122]:
df_harvey= df[df['storm']=='HARVEY']
df_idalia= df[df['storm']=='IDALIA']
df_hermine= df[df['storm']=='HERMINE']
df_matthew= df[df['storm']=='MATTHEW']
df_ian= df[df['storm']=='IAN']
df_ida= df[df['storm']=='IDA']
df_charley= df[df['storm']=='CHARLEY']

### Data Processing

In [123]:
def stations_remover(df, count):
    station_counts= df['station_id'].value_counts()
    matching_station_ids= station_counts[station_counts == count].index

    filtered_df= df[df['station_id'].isin(matching_station_ids)]

    return filtered_df

In [124]:
# df_harvey['station_id'].value_counts().value_counts() #263 is the prevalent
# df_idalia['station_id'].value_counts().value_counts() #168 is the prevalent
# df_hermine['station_id'].value_counts().value_counts() #173 is the prevalent
# df_matthew['station_id'].value_counts().value_counts() #136
# df_ian['station_id'].value_counts().value_counts() #143
# df_ida['station_id'].value_counts().value_counts() #107
# df_charley['station_id'].value_counts().value_counts() #144

df_harvey= stations_remover(df_harvey, 263)
df_idalia= stations_remover(df_idalia, 168)
df_hermine= stations_remover(df_hermine, 173)
df_matthew= stations_remover(df_matthew, 136)
df_ian= stations_remover(df_ian, 143)
df_ida= stations_remover(df_ida, 107)
df_charley= stations_remover(df_charley, 144)

In [125]:
df_harvey['station_id'].value_counts().value_counts()

count
263    254
Name: count, dtype: int64

In [126]:
df_ian['station_id'].value_counts().value_counts()

count
143    257
Name: count, dtype: int64

#### Abnormal Stations Removal

In [127]:
df_harvey.shape

(66802, 11)

In [128]:
dfs= [df_harvey, df_idalia, df_hermine, df_matthew, df_ian, df_ida, df_charley]
stations_to_remove= ['85670', '76030', '76065']

for idx, df in enumerate(dfs):
    dfs[idx] = df[~df['station_id'].isin(stations_to_remove)]

df_harvey, df_idalia, df_hermine, df_matthew, df_ian, df_ida, df_charley= dfs

In [129]:
df_harvey.shape

(66276, 11)

In [130]:
df_ian['station_id'].value_counts().value_counts()

count
143    256
Name: count, dtype: int64

In [131]:
df_ian.shape

(36608, 11)

In [132]:
## To follow a matrix format, we will isolate the 105 points around the storm surge date, so that we would have a 
## (5, 21) matrix to input to our model

#### Hurricanes' Data Subsets

In [133]:
time_utc_harvey= df_harvey['time_UTC'].sort_values().drop_duplicates().tolist()

print(len(time_utc_harvey))
print(time_utc_harvey.index('2017-08-28 12:00:00'))

# time_utc_harvey= time_utc_harvey[127:232]
time_utc_harvey= time_utc_harvey[100:250]

print(len(time_utc_harvey))

263
179
150


In [134]:
time_utc_idalia= df_idalia['time_UTC'].sort_values().drop_duplicates().tolist()

print(len(time_utc_idalia))
print(time_utc_idalia.index('2023-08-30 12:00:00'))

time_utc_idalia= time_utc_idalia[33:168]

print(len(time_utc_idalia))

168
89
135


In [135]:
time_utc_charley= df_charley['time_UTC'].sort_values().drop_duplicates().tolist()

print(len(time_utc_charley))
print(time_utc_charley.index('2004-08-13 12:00:00'))

time_utc_charley= time_utc_charley[9:144]

print(len(time_utc_charley))

144
95
135


In [136]:
time_utc_matthew= df_matthew['time_UTC'].sort_values().drop_duplicates().tolist()

print(len(time_utc_matthew))
print(time_utc_matthew.index('2016-10-05 12:00:00'))

time_utc_matthew= time_utc_matthew[1:136] 

print(len(time_utc_matthew))

136
59
135


In [116]:
time_utc_ian= df_ian['time_UTC'].sort_values().drop_duplicates().tolist()

print(len(time_utc_ian))
print(time_utc_ian.index('2022-09-29 12:00:00'))

time_utc_ian= time_utc_ian[8:143]

print(len(time_utc_ian))

143
107
135


In [137]:
time_utc_hermine= df_hermine['time_UTC'].sort_values().drop_duplicates().tolist()

print(len(time_utc_hermine))
print(time_utc_hermine.index('2016-09-02 12:00:00'))

time_utc_hermine= time_utc_hermine[38:173]

print(len(time_utc_hermine))

173
113
135


In [138]:
time_utc_ida= df_ida['time_UTC'].sort_values().drop_duplicates().tolist()

print(len(time_utc_ida))
print(time_utc_ida.index('2021-08-29 12:00:00'))

time_utc_ida= time_utc_ida[0:107]

print(len(time_utc_ida))

107
71
107


#### Dataframes Filtering

In [139]:
df_harvey= df_harvey[df_harvey['time_UTC'].isin(time_utc_harvey)]

df_harvey

,time_UTC,observed_data,forecast_data,offset,x,y,station_id,agency,storm,year,time_UTC_dt
125997,2017-08-25 05:00:00,3.315,3.154,-0.161,-70.872831,42.815647,01100870,USGS,HARVEY,2017,6060.0
125998,2017-08-25 06:00:00,4.685,4.598,-0.087,-70.872831,42.815647,01100870,USGS,HARVEY,2017,6120.0
125999,2017-08-25 07:00:00,4.945,5.028,0.083,-70.872831,42.815647,01100870,USGS,HARVEY,2017,6180.0
126000,2017-08-25 08:00:00,3.845,3.960,0.115,-70.872831,42.815647,01100870,USGS,HARVEY,2017,6240.0
126001,2017-08-25 09:00:00,2.185,2.340,0.155,-70.872831,42.815647,01100870,USGS,HARVEY,2017,6300.0
...,...,...,...,...,...,...,...,...,...,...,...
209110,2017-08-31 06:00:00,-0.169,-0.012,0.157,-72.380044,18.537800,STOFS_ptpr,IOC-UNESCO,HARVEY,2017,14760.0
209111,2017-08-31 07:00:00,-0.084,0.040,0.124,-72.380044,18.537800,STOFS_ptpr,IOC-UNESCO,HARVEY,2017,14820.0
209112,2017-08-31 08:00:00,-0.038,0.053,0.091,-72.380044,18.537800,STOFS_ptpr,IOC-UNESCO,HARVEY,2017,14880.0
209113,2017-08-31 09:00:00,0.018,0.024,0.006,-72.380044,18.537800,STOFS_ptpr,IOC-UNESCO,HARVEY,2017,14940.0


In [140]:
df_ian= df_ian[df_ian['time_UTC'].isin(time_utc_ian)]


In [141]:
df_charley= df_charley[df_charley['time_UTC'].isin(time_utc_charley)]
df_idalia= df_idalia[df_idalia['time_UTC'].isin(time_utc_idalia)]
df_matthew= df_matthew[df_matthew['time_UTC'].isin(time_utc_matthew)]
df_hermine= df_hermine[df_hermine['time_UTC'].isin(time_utc_hermine)]
df_ida= df_ida[df_ida['time_UTC'].isin(time_utc_ida)]

In [143]:
def filter_data(df, storm, threshold):
    global_mean = df['offset'].mean()
    global_std = df['offset'].std()
    threshold = 3 * global_std

    print(f"Threshold for {storm}: {threshold}")
    print('Unique stations count before filtering: ', df['station_id'].nunique())
    def filter_outliers(df):
        station_mean = df['offset'].mean()
        return df[(df['offset'] >= station_mean - threshold) & (df['offset'] <= station_mean + threshold)]
    
    filtered_df = df.groupby('station_id').apply(filter_outliers).reset_index(drop=True)

    station_counts = filtered_df['station_id'].value_counts()
    valid_stations = station_counts[station_counts >= threshold].index
    
    filtered_df = filtered_df[filtered_df['station_id'].isin(valid_stations)]

    print('Unique stations count after filtering: ', filtered_df['station_id'].nunique())
    print('-------------------------------------------------\n')
    return filtered_df

In [23]:
df_harvey= filter_data(df_harvey, 'Harvey')


Threshold for Harvey: 3.704826412698672
Unique stations count before filtering:  252
Unique stations count after filtering:  252
-------------------------------------------------



In [144]:
df_ian= filter_data(df_ian, 'Ian', 135)


Threshold for Ian: 3.4089807836847337
Unique stations count before filtering:  256
Unique stations count after filtering:  256
-------------------------------------------------



In [145]:
df_harvey= filter_data(df_harvey, 'Harvey', 135)
df_charley= filter_data(df_charley, 'Charley', 135)
df_idalia= filter_data(df_idalia, 'Idalia', 135)
df_matthew= filter_data(df_matthew, 'Matthew', 135)
df_hermine= filter_data(df_hermine, 'Hermine', 135)
df_ida= filter_data(df_ida, 'Ida', 107)

Threshold for Harvey: 3.704826412698672
Unique stations count before filtering:  252
Unique stations count after filtering:  252
-------------------------------------------------

Threshold for Charley: 5.141620976522551
Unique stations count before filtering:  108
Unique stations count after filtering:  108
-------------------------------------------------

Threshold for Idalia: 4.300995099210962
Unique stations count before filtering:  308
Unique stations count after filtering:  308
-------------------------------------------------

Threshold for Matthew: 3.036893289146885
Unique stations count before filtering:  244
Unique stations count after filtering:  244
-------------------------------------------------

Threshold for Hermine: 3.4726153120448733
Unique stations count before filtering:  270
Unique stations count after filtering:  270
-------------------------------------------------

Threshold for Ida: 4.309208107849209
Unique stations count before filtering:  269
Unique station

In [147]:
df_harvey['time_UTC'].value_counts() # each time_UTC of the 105 is repeated 246 times, the number of stations
                                     # after filtering, therefore there are not issues

time_UTC
2017-08-28 08:00:00    252
2017-08-28 19:00:00    252
2017-08-29 14:00:00    252
2017-08-29 12:00:00    252
2017-08-29 11:00:00    252
                      ... 
2017-08-25 11:00:00    248
2017-08-25 10:00:00    248
2017-08-26 11:00:00    248
2017-08-26 12:00:00    248
2017-08-25 23:00:00    248
Name: count, Length: 150, dtype: int64

In [146]:
df_ian['time_UTC'].value_counts()

time_UTC
2022-09-27 01:00:00    256
2022-09-30 09:00:00    256
2022-09-26 19:00:00    256
2022-09-30 08:00:00    256
2022-09-27 00:00:00    256
                      ... 
2022-09-26 09:00:00    252
2022-09-27 22:00:00    252
2022-09-28 23:00:00    252
2022-09-25 21:00:00    252
2022-09-27 10:00:00    252
Name: count, Length: 135, dtype: int64

#### Storm Dataframes

#### Harvey

In [148]:
# df_harvey.drop(['forecast_data', 'observed_data', 'time_UTC_dt'], axis=1, inplace= True)

df_harvey['row_num']= df_harvey.groupby('station_id').cumcount()

harvey_pivot= df_harvey.pivot_table(values='offset', index='row_num', columns='station_id', aggfunc='first')

harvey_pivot

station_id,01100870,01105876,01194750,01194796,01194815,012112296,01304200,01304562,01304746,01304920,...,9752695,9754228,9755371,9757809,9759110,9759394,9759938,STOFS_caph,STOFS_dmon,STOFS_ptpr
row_num,,,,,,,,,,,,,,,,,,,,,
0,-0.161,-3.271,-0.176,-0.232,-0.429,-0.451,-0.498,-0.243,-0.627,-0.387,...,-0.471,-0.466,-0.433,-0.449,-0.365,-0.548,-0.150,0.466,-0.054,-0.138
1,-0.087,-6.139,-0.128,-0.286,-0.539,-0.649,-0.302,-0.267,-0.636,-0.372,...,-0.404,-0.400,-0.554,-0.380,-0.410,-0.521,-0.110,0.047,-0.056,-0.043
2,0.083,-6.740,-0.176,-0.384,-0.648,-0.832,-0.173,-0.308,-0.660,-0.320,...,-0.354,-0.348,-0.638,-0.382,-0.382,-0.438,-0.091,-0.411,-0.051,-0.070
3,0.115,-4.317,-0.357,-0.596,-0.841,-1.034,-0.358,-0.383,-0.647,-0.240,...,-0.319,-0.422,-0.637,-0.391,-0.383,-0.417,-0.312,-0.768,-0.038,0.003
4,0.155,-2.182,-0.384,-0.734,-1.019,-1.168,-0.668,-0.530,-0.681,-0.192,...,-0.331,-0.329,-0.508,-0.341,-0.367,-0.303,-0.277,-0.921,0.006,-0.077
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,-0.312,NaN,-0.649,-0.788,-0.888,-0.783,-0.745,-1.195,-1.150,-1.242,...,-0.330,-0.318,-0.155,-0.183,-0.236,-0.451,-0.216,0.127,-0.146,0.157
146,-0.505,NaN,-0.550,-0.705,-0.885,-0.601,-0.888,-0.872,-1.127,-1.199,...,-0.342,-0.316,-0.184,-0.185,-0.321,-0.438,-0.363,0.221,-0.197,0.124
147,-0.608,NaN,-0.565,-0.682,-0.856,-0.519,-0.865,-0.617,-1.120,-1.083,...,-0.336,-0.233,-0.233,-0.164,-0.318,-0.411,-0.347,0.222,-0.161,0.091


In [43]:
harvey_pivot = harvey_pivot.dropna(axis=1)

harvey_pivot

station_id,01100870,01194750,01194796,01194815,012112296,01304200,01304562,01304746,01304920,01306402,...,9752695,9754228,9755371,9757809,9759110,9759394,9759938,STOFS_caph,STOFS_dmon,STOFS_ptpr
row_num,,,,,,,,,,,,,,,,,,,,,
0,-0.161,-0.176,-0.232,-0.429,-0.451,-0.498,-0.243,-0.627,-0.387,-0.288,...,-0.471,-0.466,-0.433,-0.449,-0.365,-0.548,-0.150,0.466,-0.054,-0.138
1,-0.087,-0.128,-0.286,-0.539,-0.649,-0.302,-0.267,-0.636,-0.372,-0.335,...,-0.404,-0.400,-0.554,-0.380,-0.410,-0.521,-0.110,0.047,-0.056,-0.043
2,0.083,-0.176,-0.384,-0.648,-0.832,-0.173,-0.308,-0.660,-0.320,-0.340,...,-0.354,-0.348,-0.638,-0.382,-0.382,-0.438,-0.091,-0.411,-0.051,-0.070
3,0.115,-0.357,-0.596,-0.841,-1.034,-0.358,-0.383,-0.647,-0.240,-0.320,...,-0.319,-0.422,-0.637,-0.391,-0.383,-0.417,-0.312,-0.768,-0.038,0.003
4,0.155,-0.384,-0.734,-1.019,-1.168,-0.668,-0.530,-0.681,-0.192,-0.282,...,-0.331,-0.329,-0.508,-0.341,-0.367,-0.303,-0.277,-0.921,0.006,-0.077
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,-0.312,-0.649,-0.788,-0.888,-0.783,-0.745,-1.195,-1.150,-1.242,-1.072,...,-0.330,-0.318,-0.155,-0.183,-0.236,-0.451,-0.216,0.127,-0.146,0.157
146,-0.505,-0.550,-0.705,-0.885,-0.601,-0.888,-0.872,-1.127,-1.199,-1.092,...,-0.342,-0.316,-0.184,-0.185,-0.321,-0.438,-0.363,0.221,-0.197,0.124
147,-0.608,-0.565,-0.682,-0.856,-0.519,-0.865,-0.617,-1.120,-1.083,-1.123,...,-0.336,-0.233,-0.233,-0.164,-0.318,-0.411,-0.347,0.222,-0.161,0.091


In [149]:
df_ian['row_num']= df_ian.groupby('station_id').cumcount()
ian_pivot= df_ian.pivot_table(values='offset', index='row_num', columns='station_id', aggfunc='first')
ian_pivot = ian_pivot.dropna(axis=1)

ian_pivot

station_id,01100870,01194750,01194796,01194815,01302600,01304200,01304650,01304746,01304920,01306402,...,8779749,9751364,9751381,9751401,9752235,9752695,9755371,9759110,9759394,9759938
row_num,,,,,,,,,,,,,,,,,,,,,
0,0.300,0.192,0.233,-0.046,-3.891,0.360,0.348,0.028,0.308,0.177,...,-0.789,-0.580,-0.201,-0.464,-0.159,-0.361,-0.205,-0.428,-0.374,-0.168
1,0.031,0.216,0.213,0.050,-3.969,0.250,0.024,-0.239,0.436,0.336,...,-0.829,-0.620,-0.246,-0.467,-0.242,-0.386,-0.156,-0.411,-0.348,-0.142
2,-0.948,0.496,0.429,0.342,-3.046,0.208,0.150,-0.314,0.588,0.346,...,-0.789,-0.621,-0.250,-0.506,-0.293,-0.404,-0.204,-0.356,-0.418,-0.289
3,-1.433,0.614,0.610,0.515,-0.772,0.090,0.141,-0.371,0.635,0.322,...,-0.797,-0.649,-0.367,-0.563,-0.268,-0.411,-0.234,-0.473,-0.467,-0.256
4,-1.427,0.452,0.408,0.275,1.715,-0.105,0.072,-0.207,0.485,0.254,...,-0.735,-0.585,-0.425,-0.558,-0.369,-0.381,-0.306,-0.541,-0.526,-0.273
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,0.675,0.137,0.018,-0.280,0.130,-0.191,-0.458,0.045,-0.268,-0.358,...,-1.478,-0.555,-0.461,-0.524,-0.236,-0.307,-0.501,-0.507,-0.331,-0.565
131,-0.130,-0.057,-0.191,-0.497,-0.120,-0.179,-0.342,0.279,-0.277,-0.356,...,-1.445,-0.515,-0.363,-0.441,-0.282,-0.246,-0.612,-0.467,-0.355,-0.652
132,-0.186,-0.232,-0.316,-0.731,-0.929,-0.249,-0.322,0.335,-0.273,-0.242,...,-1.428,-0.472,-0.294,-0.406,-0.284,-0.222,-0.605,-0.409,-0.404,-0.677


In [150]:
df_idalia['row_num']= df_idalia.groupby('station_id').cumcount()
idalia_pivot= df_idalia.pivot_table(values='offset', index='row_num', columns='station_id', aggfunc='first')
idalia_pivot = idalia_pivot.dropna(axis=1)

df_matthew['row_num']= df_matthew.groupby('station_id').cumcount()
matthew_pivot= df_matthew.pivot_table(values='offset', index='row_num', columns='station_id', aggfunc='first')
matthew_pivot = matthew_pivot.dropna(axis=1)

df_hermine['row_num']= df_hermine.groupby('station_id').cumcount()
hermine_pivot= df_hermine.pivot_table(values='offset', index='row_num', columns='station_id', aggfunc='first')
hermine_pivot = hermine_pivot.dropna(axis=1)

df_charley['row_num']= df_charley.groupby('station_id').cumcount()
charley_pivot= df_charley.pivot_table(values='offset', index='row_num', columns='station_id', aggfunc='first')
charley_pivot = charley_pivot.dropna(axis=1)

df_ida['row_num']= df_ida.groupby('station_id').cumcount()
ida_pivot= df_ida.pivot_table(values='offset', index='row_num', columns='station_id', aggfunc='first')
ida_pivot = ida_pivot.dropna(axis=1)


In [29]:
unique_stations = df_harvey[['station_id', 'x', 'y']].drop_duplicates()
unique_df = pd.DataFrame(unique_stations, columns=['station_id', 'x', 'y'])

print(len(unique_df))

unique_df = unique_df[unique_df['station_id'].isin(harvey_pivot.columns)]

unique_df

252


,station_id,x,y
0,01100870,-70.872831,42.815647
150,01105876,-70.622534,41.941770
261,01194750,-72.384367,41.351483
411,01194796,-72.345917,41.312598
561,01194815,-72.349278,41.281333
...,...,...,...
36918,9759394,-67.162444,18.218833
37068,9759938,-67.940285,18.089289
37218,STOFS_caph,-72.193371,19.759303
37368,STOFS_dmon,-70.653100,39.296400


In [45]:
df_harvey_subset = df_harvey[df_harvey['station_id'].isin(harvey_pivot.columns)]

df_harvey_subset['station_id'].nunique()

246

In [46]:
df_harvey_subset.to_csv("df_harvey_subset.csv", index=False)

In [151]:
df_ian_subset = df_ian[df_ian['station_id'].isin(ian_pivot.columns)]
df_ian_subset['station_id'].nunique()

df_ian_subset.to_csv("df_ian_subset.csv", index=False)

In [152]:
df_idalia_subset = df_idalia[df_idalia['station_id'].isin(idalia_pivot.columns)]
df_idalia_subset['station_id'].nunique()

df_idalia_subset.to_csv("df_idalia_subset.csv", index=False)

In [153]:
df_matthew_subset = df_matthew[df_matthew['station_id'].isin(matthew_pivot.columns)]
df_matthew_subset['station_id'].nunique()

df_matthew_subset.to_csv("df_matthew_subset.csv", index=False)

In [154]:
df_hermine_subset = df_hermine[df_hermine['station_id'].isin(hermine_pivot.columns)]
df_hermine_subset['station_id'].nunique()

df_hermine_subset.to_csv("df_hermine_subset.csv", index=False)

In [155]:
df_charley_subset = df_charley[df_charley['station_id'].isin(charley_pivot.columns)]
df_charley_subset['station_id'].nunique()

df_charley_subset.to_csv("df_charley_subset.csv", index=False)

In [156]:
df_ida_subset = df_ida[df_ida['station_id'].isin(ida_pivot.columns)]
df_ida_subset['station_id'].nunique()

df_ida_subset.to_csv("df_ida_subset.csv", index=False)